# Predictive Modelling of Breast Cancer Diagnosis Using Machine Learning Techniques

**Researcher:** Ososanwo Yetunde Oluwabusolami  
**Dataset:** Breast Cancer Coimbra Dataset (BCCD), UCI ML Repository (DOI: 10.24432/C52P59)  
**Pipeline:** scikit-learn unified Pipeline + nested stratified cross-validation  
**Classifiers:** Logistic Regression, SVM (RBF), Decision Tree, Random Forest, MLP-ANN

This notebook operationalises the methodology specified in Chapter 3 of the project. It is reproducible from end to end with the supplied random seed (`random_state=42`).

## 1. Environment & Dependencies

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# scikit-learn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Aesthetics
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'serif'

print('Environment ready. NumPy:', np.__version__, '| Pandas:', pd.__version__)

## 2. Data Loading & Schema Verification

The BCCD comprises 116 observations and 10 variables: 9 predictors + 1 binary target (`Classification`).

In [ ]:
# Load dataset
df = pd.read_csv('dataR2.csv')

# Schema verification
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Missing values:', df.isnull().sum().sum())
print('\nClass distribution:')
print(df['Classification'].value_counts())
df.head()

## 3. Exploratory Data Analysis (§3.5)

In [ ]:
# 3.5.1 Descriptive statistics
desc = df.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2)
print('Table 3.3: Descriptive statistics')
desc

In [ ]:
# 3.5.2 Skewness
skew = df.drop(columns='Classification').skew().sort_values(ascending=False).round(2)
print('Skewness by feature:')
print(skew)

# Distributional plots
features = df.columns.drop('Classification')
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.flat, features):
    sns.histplot(data=df, x=col, hue='Classification', kde=True, ax=ax,
                 palette=['#2E75B6', '#C00000'], alpha=0.6)
    ax.set_title(col)
plt.suptitle('Figure 3.1: Feature distributions by class', y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5.3 Bivariate correlations with target
corr_target = (df.corr()['Classification']
                 .drop('Classification')
                 .sort_values(key=abs, ascending=False)
                 .round(3))
print('Table 3.4: Pearson correlations with target')
print(corr_target)

# Full correlation matrix heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Figure 3.2: Pearson correlation matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5.5 Outlier diagnostics via boxplots
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.flat, features):
    sns.boxplot(data=df, x='Classification', y=col, hue='Classification',
                ax=ax, palette=['#2E75B6', '#C00000'], legend=False)
    ax.set_xticklabels(['Control', 'Patient'])
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Figure 3.3: Class-stratified boxplots', y=1.00, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Preprocessing & Partitioning (§§3.6–3.7)

In [ ]:
# 3.6.1 Re-encode target: 1=control,2=patient -> 0=control,1=patient
X = df.drop(columns='Classification').copy()
y = (df['Classification'] == 2).astype(int)

print('Feature matrix X:', X.shape)
print('Target y class balance:', y.value_counts().to_dict())
print(f'Positive class (patient) proportion: {y.mean():.3f}')

In [ ]:
# 3.7 Stratified 80:20 train–test partition with fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

print(f'Train set: n = {len(X_train)} (patients: {y_train.sum()}, controls: {(y_train==0).sum()})')
print(f'Test  set: n = {len(X_test)} (patients: {y_test.sum()}, controls: {(y_test==0).sum()})')
print(f'Stratification preserved: train ratio = {y_train.mean():.3f}, test ratio = {y_test.mean():.3f}')

## 5. Pipeline Construction (§3.9)

In [ ]:
def build_pipeline(classifier):
    """Construct a unified preprocessing + classification Pipeline.

    Imputation -> Scaling -> Classification. Encapsulating these
    steps prevents test-set information leaking into preprocessing.
    """
    return Pipeline(steps=[
        ('imputer',    SimpleImputer(strategy='median')),
        ('scaler',     StandardScaler()),
        ('classifier', classifier),
    ])

# Cross-validation splitters (3-fold inner for tuning, 5-fold outer for CV scores)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Pipeline factory and CV splitters ready.')

## 6. Classifier Specifications & Hyperparameter Grids (§3.8)

In [ ]:
classifiers = {
    'Logistic Regression': {
        'estimator': LogisticRegression(
            penalty='l2', solver='lbfgs', max_iter=1000, random_state=SEED
        ),
        'param_grid': {
            'classifier__C': [0.01, 0.1, 1, 10, 100],
        },
    },
    'SVM (RBF)': {
        'estimator': SVC(kernel='rbf', probability=True, random_state=SEED),
        'param_grid': {
            'classifier__C': [0.1, 1, 10, 100],
            'classifier__gamma': ['scale', 'auto', 0.01, 0.1, 1],
        },
    },
    'Decision Tree': {
        'estimator': DecisionTreeClassifier(random_state=SEED),
        'param_grid': {
            'classifier__max_depth': [3, 5, 7, 10, None],
            'classifier__min_samples_split': [2, 5, 10],
            'classifier__min_samples_leaf':  [1, 2, 5],
        },
    },
    'Random Forest': {
        'estimator': RandomForestClassifier(random_state=SEED, n_jobs=-1),
        'param_grid': {
            'classifier__n_estimators': [100, 200, 500],
            'classifier__max_depth':    [5, 10, None],
            'classifier__max_features': ['sqrt', 'log2'],
        },
    },
    'MLP-ANN': {
        'estimator': MLPClassifier(max_iter=1000, random_state=SEED),
        'param_grid': {
            'classifier__hidden_layer_sizes': [(50,), (100,), (50, 25), (100, 50)],
            'classifier__activation': ['relu', 'tanh'],
            'classifier__solver':     ['adam'],
            'classifier__alpha':      [0.0001, 0.001, 0.01],
        },
    },
}

for name, spec in classifiers.items():
    grid_size = 1
    for v in spec['param_grid'].values():
        grid_size *= len(v)
    print(f'{name:22s} | grid size: {grid_size:3d}')

## 7. Nested Cross-Validation: Tuning + Evaluation

In [ ]:
def evaluate_classifier(name, spec, X_train, y_train, X_test, y_test):
    """Tune via inner CV, refit, then evaluate on the held-out test set."""
    pipe = build_pipeline(spec['estimator'])

    grid = GridSearchCV(
        pipe, param_grid=spec['param_grid'],
        cv=inner_cv, scoring='roc_auc', n_jobs=-1, refit=True
    )
    grid.fit(X_train, y_train)
    best = grid.best_estimator_

    # Outer CV scores on TRAINING data (for statistical comparison)
    cv_auc = cross_val_score(best, X_train, y_train, cv=outer_cv, scoring='roc_auc')

    # Held-out test-set metrics
    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    metrics = {
        'Classifier':   name,
        'Best Params':  grid.best_params_,
        'CV AUC (mean)':  cv_auc.mean(),
        'CV AUC (std)':   cv_auc.std(),
        'CV AUC scores':  cv_auc,
        'Test Accuracy':  accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred, zero_division=0),
        'Test Recall':    recall_score(y_test, y_pred),
        'Test Specificity': specificity,
        'Test F1':        f1_score(y_test, y_pred),
        'Test AUC':       roc_auc_score(y_test, y_proba),
        'Confusion':      cm,
        'y_proba':        y_proba,
        'fitted':         best,
    }
    return metrics

results = {}
for name, spec in classifiers.items():
    print(f'Evaluating {name} ...', end=' ')
    results[name] = evaluate_classifier(name, spec, X_train, y_train, X_test, y_test)
    print(f"AUC = {results[name]['Test AUC']:.3f}")

print('\nAll five classifiers evaluated.')

## 8. Comparative Performance Summary (§3.10)

In [ ]:
summary = pd.DataFrame([{
    'Classifier':    m['Classifier'],
    'CV AUC (mean)': round(m['CV AUC (mean)'], 4),
    'CV AUC (std)':  round(m['CV AUC (std)'], 4),
    'Accuracy':      round(m['Test Accuracy'], 4),
    'Precision':     round(m['Test Precision'], 4),
    'Recall (Sens.)':round(m['Test Recall'], 4),
    'Specificity':   round(m['Test Specificity'], 4),
    'F1':            round(m['Test F1'], 4),
    'AUC-ROC':       round(m['Test AUC'], 4),
} for m in results.values()])

summary = summary.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
print('Table 4.1: Comparative classifier performance')
summary

In [ ]:
# Bar plot of held-out test-set metrics
plot_metrics = ['Accuracy', 'Precision', 'Recall (Sens.)', 'Specificity', 'F1', 'AUC-ROC']
plot_df = summary.set_index('Classifier')[plot_metrics]

fig, ax = plt.subplots(figsize=(13, 6))
plot_df.plot(kind='bar', ax=ax, colormap='Blues', edgecolor='black', width=0.85)
ax.set_title('Figure 4.1: Held-out test set performance across classifiers',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05); ax.set_xlabel('')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
ax.axhline(0.5, color='red', ls=':', lw=1, alpha=0.6)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, m) in zip(axes, results.items()):
    ConfusionMatrixDisplay(m['Confusion'], display_labels=['Control', 'Patient']) \
        .plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"{name}\nAUC={m['Test AUC']:.3f}", fontsize=10)
plt.suptitle('Figure 4.2: Confusion matrices on held-out test set',
             y=1.05, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves overlaid
plt.figure(figsize=(9, 7))
for name, m in results.items():
    fpr, tpr, _ = roc_curve(y_test, m['y_proba'])
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={m['Test AUC']:.3f})")
plt.plot([0, 1], [0, 1], color='grey', ls='--', alpha=0.6)
plt.xlabel('False Positive Rate (1 − Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Figure 4.3: ROC curves — all classifiers (held-out test set)',
          fontsize=14, fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Statistical Comparison Across Classifiers (§3.10.2)

Pairwise Wilcoxon signed-rank tests on the 5-fold cross-validation AUC scores. Bonferroni-corrected α ≈ .005 (across 10 pairwise comparisons).

In [ ]:
names = list(results.keys())
pvals = pd.DataFrame(index=names, columns=names, dtype=float)
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i == j:
            pvals.loc[a, b] = np.nan
        else:
            sa = results[a]['CV AUC scores']
            sb = results[b]['CV AUC scores']
            try:
                _, pv = stats.wilcoxon(sa, sb)
            except ValueError:
                pv = 1.0  # identical samples
            pvals.loc[a, b] = round(pv, 4)

print('Pairwise Wilcoxon p-values on CV AUC scores (Bonferroni-corrected α ≈ .005):')
pvals

## 10. Best Model Selection & Feature Importance (§3.11)

In [ ]:
best_name = summary.iloc[0]['Classifier']
best = results[best_name]['fitted']
print(f'Best-performing model: {best_name}')
print(f'  CV AUC: {results[best_name]["CV AUC (mean)"]:.4f} ± {results[best_name]["CV AUC (std)"]:.4f}')
print(f'  Test AUC: {results[best_name]["Test AUC"]:.4f}')
print(f'  Test Recall (Sensitivity): {results[best_name]["Test Recall"]:.4f}')
print(f'  Test Specificity: {results[best_name]["Test Specificity"]:.4f}')

In [ ]:
# Permutation importance — model-agnostic, works for every classifier family
perm = permutation_importance(
    best, X_test, y_test, n_repeats=30, random_state=SEED,
    scoring='roc_auc', n_jobs=-1
)
imp = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': perm.importances_mean,
    'StdDev':     perm.importances_std,
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print(f'Table 4.2: Permutation feature importance for {best_name}')
print(imp.round(4))

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=imp, x='Importance', y='Feature',
                 hue='Feature', palette='Blues_r', legend=False)
ax.errorbar(x=imp['Importance'], y=range(len(imp)),
            xerr=imp['StdDev'], fmt='none', color='black', capsize=3)
plt.title(f'Figure 4.4: Permutation feature importance — {best_name}',
          fontsize=13, fontweight='bold')
plt.xlabel('Drop in AUC-ROC when feature is permuted')
plt.tight_layout()
plt.show()

## 11. SHAP Interpretability (Optional Extension)

If `shap` is installed, the cell below produces global and per-instance Shapley value explanations for the best-performing model.

In [ ]:
try:
    import shap

    # Transform test set through the imputer + scaler steps of the best pipeline
    X_test_transformed = best[:-1].transform(X_test)
    estimator = best.named_steps['classifier']

    # Tree-based estimators -> TreeExplainer; everything else -> KernelExplainer
    if hasattr(estimator, 'estimators_') or isinstance(estimator, DecisionTreeClassifier):
        explainer = shap.TreeExplainer(estimator)
    else:
        explainer = shap.KernelExplainer(estimator.predict_proba,
                                         shap.sample(X_test_transformed, 30, random_state=SEED))

    shap_vals = explainer.shap_values(X_test_transformed)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]  # positive class

    shap.summary_plot(shap_vals, X_test_transformed, feature_names=X.columns.tolist(),
                      show=False)
    plt.title(f'Figure 4.5: SHAP summary plot — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('shap not installed. Run `pip install shap` to enable this section.')
except Exception as e:
    print(f'SHAP analysis skipped: {e}')

## 12. Clinical Plausibility Assessment (§3.11.3)

Triangulation table mapping empirical feature importance to the prior biomedical literature.

In [ ]:
literature = {
    'Resistin':    ('Strong support', 'Chu et al. 2024; Patrício et al. 2018'),
    'Glucose':     ('Strong support', 'Patrício et al. 2018'),
    'Insulin':     ('Strong support', 'Chu et al. 2024 (IGF-1 axis)'),
    'HOMA':        ('Strong support', 'Naranjo-Torres et al. 2024'),
    'BMI':         ('Strong support', 'Naranjo-Torres et al. 2024 (post-menopausal)'),
    'Leptin':      ('Strong support', 'Chu et al. 2024 (JAK/STAT, MAPK)'),
    'MCP.1':       ('Moderate support', 'Chu et al. 2024 (tumour microenvironment)'),
    'Age':         ('Strong support', 'Giaquinto et al. 2024'),
    'Adiponectin': ('Contested / inverse', 'Naranjo-Torres et al. 2024'),
}

plausibility = imp.copy()
plausibility['Biomedical Support'] = plausibility['Feature'].map(lambda f: literature.get(f, ('Unknown',''))[0])
plausibility['Citation']          = plausibility['Feature'].map(lambda f: literature.get(f, ('Unknown',''))[1])

print(f'Table 4.3: Clinical plausibility assessment for {best_name}')
plausibility[['Feature', 'Importance', 'Biomedical Support', 'Citation']].round(4)

## 13. Save Best Model & Results

In [ ]:
import joblib, json
joblib.dump(best, f'best_model_{best_name.replace(" ", "_").replace("(", "").replace(")", "")}.pkl')
summary.to_csv('classifier_performance_summary.csv', index=False)
imp.to_csv('feature_importance.csv', index=False)
plausibility.to_csv('plausibility_assessment.csv', index=False)
print('Best model and result tables saved to working directory.')

---

**End of notebook.** Outputs serialised: best-model pickle, performance summary CSV, feature-importance CSV, and clinical plausibility CSV. These artefacts feed directly into Chapter Four (Results) and Chapter Five (Discussion).